In [ ]:
import librosa
import matplotlib.pyplot as plt
from src.utils.counting import *
import numpy as np
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
AUDIO_DIR = "path/to/detected/wav/trill/segments/"
DATA_DIR = "path/to/annotation/file/"

ground_truth_annotation_file = "annotations_trills.csv"
prediction_annotation_file = "final_virtuosity_dataset_v3.csv"

df_gt = pd.read_csv(os.path.join(DATA_DIR, ground_truth_annotation_file))
df_pred = pd.read_csv(os.path.join(DATA_DIR, prediction_annotation_file))

df_inter = pd.merge(
    df_gt[["file_name_radical", "segment_id", "trill_min_freq", "trill_max_freq", "num_trills"]],
    df_pred[["file_name_radical", "segment_id", "f_min", "f_max", "f_min_podos", "f_max_podos", "time_start", "time_end"]],
    on=["file_name_radical", "segment_id"],
    suffixes=("_true", "_pred")
)

df_list_save = pd.merge(
    df_gt[["file_name_radical", "segment_id"]],
    df_pred,
    on=["file_name_radical", "segment_id"],
)

In [ ]:
def compute_rms(row):
    audio_filename = row["file_name_radical"]   
    y, sr = librosa.load(os.path.join(AUDIO_DIR, audio_filename), sr=None)
    start = row["time_start"]
    end = row["time_end"]

    start_sample = int(start * sr)
    end_sample = int(end * sr)
    segment = y[start_sample:end_sample]
    rms = librosa.feature.rms(y=segment, frame_length=256, hop_length=64).mean()
    return rms

    

In [ ]:
df_list_save.to_csv(os.path.join(DATA_DIR, "comparison_true_pred.csv"), index=False)

In [ ]:
df_inter["bandwidth_true"] = df_inter["trill_max_freq"] - df_inter["trill_min_freq"]
df_inter["bandwidth_pred"] = df_inter["f_max"] - df_inter["f_min"]
df_inter["bandwidth_podos"] = df_inter["f_max_podos"] - df_inter["f_min_podos"]
df_inter["error_bandwidth_custom"] = df_inter["bandwidth_pred"] - df_inter["bandwidth_true"]
df_inter["error_bandwidth_podos"] = df_inter["bandwidth_podos"] - df_inter["bandwidth_true"]
df_inter["rms"] = df_inter.apply(compute_rms, axis=1)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots()

# Add traces
fig.add_trace(
    go.Scatter(x=df_inter["bandwidth_true"], 
               y=df_inter["bandwidth_pred"], 
               name="Prediction custom solution",
               mode="markers", 
               hovertext=df_inter["file_name_radical"] + " seg" + df_inter["segment_id"].astype(str)),
)

fig.add_trace(
    go.Scatter(x=df_inter["bandwidth_true"], 
               y=df_inter["bandwidth_podos"],
               name="Prediction Podos solution",
               mode="markers",
               hovertext=df_inter["file_name_radical"] + " seg" + df_inter["segment_id"].astype(str)),
)

fig.add_trace(
    go.Scatter(x=df_inter["bandwidth_true"], 
               y=df_inter["bandwidth_true"],
               name="Ideal (y=x)",
               mode="lines",
               line=dict(color="black", dash="dash"))
)

# Add figure title
fig.update_layout(
    title_text="Bandwidth comparison between true values and predictions (custom vs predictions)",
    width=1200,
    height=800
)

# Set x-axis title
fig.update_xaxes(title_text="Ground truth bandwidth (Hz)")

# Set y-axes titles
fig.update_yaxes(title_text="<b>primary</b> Predicted bandwidth (Hz)", secondary_y=False)

fig.show()

In [ ]:
all_vals = pd.concat([df_inter["error_bandwidth_custom"], 
                      df_inter["error_bandwidth_podos"]]).dropna()
bin_min  = all_vals.min()
bin_max  = all_vals.max()
bin_size = (bin_max - bin_min) / 50  # 50 bins, ajuste à ta convenance

bins = dict(start=bin_min, end=bin_max, size=bin_size)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=df_inter["error_bandwidth_custom"],
    name="Custom solution error",
    xbins=bins,
    histnorm="probability"
))
fig.add_trace(go.Histogram(
    x=df_inter["error_bandwidth_podos"],
    name="Podos solution error",
    xbins=bins,
    histnorm="probability"
))
fig.update_layout(barmode="overlay", title="Distribution of bandwidth prediction errors")
fig.update_traces(opacity=0.75)
fig.update_xaxes(title_text="Prediction error (Hz)")
fig.update_yaxes(title_text="Probability")
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots()

# Add traces
fig.add_trace(
    go.Scatter(x=df_inter["rms"], 
               y=df_inter["error_bandwidth_custom"], 
               name="Prediction custom solution",
               mode="markers", 
               hovertext=df_inter["file_name_radical"] + " seg" + df_inter["segment_id"].astype(str)),
)

fig.add_trace(
    go.Scatter(x=df_inter["rms"], 
               y=df_inter["error_bandwidth_podos"],
               name="Prediction Podos solution",
               mode="markers",
               hovertext=df_inter["file_name_radical"] + " seg" + df_inter["segment_id"].astype(str)),
)

# fig.add_trace(
#     go.Scatter(x=df_inter["rms"], 
#                y=df_inter["bandwidth_true"],
#                name="Ideal (y=x)",
#                mode="lines",
#                line=dict(color="black", dash="dash"))
# )

# Add figure title
fig.update_layout(
    title_text="Bandwidth comparison between true values and predictions (custom vs predictions)",
    width=1200,
    height=800
)

# Set x-axis title
fig.update_xaxes(title_text="RMS of the segment")

# Set y-axes titles
fig.update_yaxes(title_text="<b>primary</b> Prediction Error (Hz)", secondary_y=False)

fig.show()